In [7]:
from jaxcmr.memorysearch import (
    BaseCMR,
    InstanceCMR,
    uniform_presentations_data_likelihood,
    variable_presentations_data_likelihood,
    predict_and_simulate_trial,
    predict_and_simulate_pres_and_trial,
    log_likelihood,
    start_retrieving,
    experience,
    trial_item_count,
    trial_list_length,
)

from jaxcmr.evaluation import extract_objective_data
from jaxcmr.datasets import load_data, generate_trial_mask
import numpy as np
from jax import numpy as jnp, jit, lax, vmap, disable_jit

data_path = '../data/{}.h5'
data = load_data(data_path.format('LohnasKahana2014'))

trial_query = "data['list_type'] != -1"
trial_mask = generate_trial_mask(data, trial_query)
trials, list_lengths, presentations, pres_string_ids, has_repetitions = extract_objective_data(data, trial_mask)

In [5]:
parameters = {
        "encoding_drift_rate": 0.8016327576683261,
        "delay_drift_rate": 0.9966411723460118,
        "start_drift_rate": 0.051123130268380085,
        "recall_drift_rate": 0.8666706252504806,
        "shared_support": 0.016122091797498662,
        "item_support": 0.8877852952105489,
        "learning_rate": 0.10455606050373444,
        "primacy_scale": 33.57091895097917,
        "primacy_decay": 1.57091895097917,
        "stop_probability_scale": 0.0034489993376706257,
        "stop_probability_growth": 0.3779780110633191,
        "choice_sensitivity": 1.0,
        "mcf_trace_sensitivity": 1.0,
    }

In [11]:
item_counts = vmap(trial_item_count)(presentations[0])
item_counts

Array([34, 40, 20, ..., 20, 20, 40], dtype=int32)

In [12]:
jnp.unique(item_counts)

Array([20, 34, 40], dtype=int32)

In [22]:
def variable_presentations_data_likelihood(
    model_init,
    presentations,
    trials,
    parameters
):
    
    functions = []
    item_counts = vmap(trial_item_count)(presentations)
    unique_item_counts = jnp.unique(item_counts).tolist()
    for item_count in unique_item_counts:

        item_count_presentations = presentations[item_counts == item_count]
        item_count_trials = trials[item_counts == item_count]

        functions.append(
            lambda parameters: vmap(
            predict_and_simulate_pres_and_trial, in_axes=(None, None, 0, 0, None))(
                model_init,
                item_count,
                item_count_presentations,
                item_count_trials,
                parameters,
            )[1])
        
    @jit
    def f(parameters):
        return log_likelihood(lax.map(
            lambda parameters: functions[unique_item_counts.index(item_count)](parameters),
            item_counts
        ))
    
    return f

In [23]:
loss = variable_presentations_data_likelihood(
    InstanceCMR.create,
    presentations[0],
    trials[0],
    parameters
)

In [24]:
loss(parameters)

NotFoundLookupError: For function `predict_and_simulate_pres_and_trial`, `(<bound method InstanceCMR.create of <class 'jaxcmr.memorysearch.InstanceCMR.InstanceCMR'>>, 40, Traced<ShapedArray(int32[40])>with<DynamicJaxprTrace(level=4/0)>, Traced<ShapedArray(int32[40])>with<DynamicJaxprTrace(level=4/0)>, Traced<ShapedArray(int32[])>with<DynamicJaxprTrace(level=4/0)>)` could not be resolved.